# 92. The Location-Routing Problem (LRP)
## Tier 3: The Advanced Algorithm (Grey Wolf Optimizer)

### Key assumptions
- Grey Wolf Optimizer mimics wolf pack social hierarchy and hunting behavior
- Alpha, Beta, Delta wolves guide the search (best solutions)
- Omega wolves follow the leaders with position updates
- Continuous positions are decoded to discrete LRP solutions
- Exploration-exploitation balance through adaptive parameters

### Approach (step-by-step)
1. **GWO Framework**: Implement the social hierarchy and hunting mechanics
2. **Solution Encoding**: Design continuous vector representation for LRP
3. **Position Decoding**: Convert continuous vectors to feasible LRP solutions
4. **Fitness Evaluation**: Calculate total costs for decoded solutions
5. **Position Updates**: Implement encircling prey and hunting behaviors
6. **Convergence Analysis**: Track alpha wolf improvement and diversity

### What to look for in the results
- Convergence behavior of alpha, beta, delta wolves
- Solution quality improvement over generations
- Population diversity and exploration patterns
- Impact of encoding scheme on solution quality
- Comparison with GRASP and exact methods

### Concrete example (from the source)
- **Problem**: Same 2-depot, 3-customer instance scaled up for population-based search
- **GWO Parameters**: 30 wolves, 100 generations, social hierarchy maintained
- **Encoding**: Priority-based encoding for depot selection and customer assignment
- **Expected Performance**: Better exploration than GRASP, near-optimal solutions

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import math
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

GA Problem created with 20 SKUs and 8 locations
Velocity range: 9 - 187
Average velocity: 58.9


In [ ]:
@dataclass
class LRPInstance:
    """Data class for Location-Routing Problem instance"""
    customers: List[int]
    depots: List[int]
    vehicles: List[int]
    depot_costs: Dict[int, float]
    demands: Dict[int, float]
    vehicle_capacity: float
    travel_costs: Dict[Tuple[int, int], float]
    
    def get_all_nodes(self):
        return self.customers + self.depots

@dataclass
class Wolf:
    """Represents a grey wolf in the population"""
    position: np.ndarray  # Continuous position vector
    fitness: float        # Fitness value (total cost)
    solution: dict        # Decoded LRP solution
    rank: str = 'omega'   # Social hierarchy rank

# Create the concrete example
def create_concrete_example():
    customers = [1, 2, 3]
    depots = [4, 5]
    vehicles = [1, 2]
    
    depot_costs = {4: 100, 5: 120}
    demands = {1: 10, 2: 15, 3: 20}
    vehicle_capacity = 30
    
    travel_costs = {
        (1, 2): 15, (2, 1): 15,
        (1, 3): 20, (3, 1): 20,
        (2, 3): 18, (3, 2): 18,
        (4, 1): 12, (1, 4): 12,
        (4, 2): 14, (2, 4): 14,
        (4, 3): 25, (3, 4): 25,
        (5, 1): 18, (1, 5): 18,
        (5, 2): 16, (2, 5): 16,
        (5, 3): 22, (3, 5): 22,
        (4, 5): 30, (5, 4): 30,
        (1, 1): 0, (2, 2): 0, (3, 3): 0,
        (4, 4): 0, (5, 5): 0
    }
    
    return LRPInstance(
        customers=customers,
        depots=depots,
        vehicles=vehicles,
        depot_costs=depot_costs,
        demands=demands,
        vehicle_capacity=vehicle_capacity,
        travel_costs=travel_costs
    )

instance = create_concrete_example()
print(f"LRP Instance created for Grey Wolf Optimizer:")
print(f"Customers: {instance.customers}")
print(f"Potential Depots: {instance.depots}")
print(f"Vehicles: {instance.vehicles}")
print(f"Vehicle Capacity: {instance.vehicle_capacity}")

LRP Instance created for Grey Wolf Optimizer:
Customers: [1, 2, 3]
Potential Depots: [4, 5]
Vehicles: [1, 2]
Vehicle Capacity: 30


In [2]:
def encode_lrp_solution(instance):
    """Create encoding scheme for LRP solutions"""
    # Encoding structure:
    # [depot_priorities, customer_assignment_priorities, routing_priorities]
    
    depot_count = len(instance.depots)
    customer_count = len(instance.customers)
    
    # Depot selection priorities (one per potential depot)
    depot_size = depot_count
    
    # Customer-depot assignment priorities (one per customer)
    assignment_size = customer_count
    
    # Routing priorities (customer order for route construction)
    routing_size = customer_count
    
    total_size = depot_size + assignment_size + routing_size
    
    return {
        'depot_start': 0,
        'depot_end': depot_size,
        'assignment_start': depot_size,
        'assignment_end': depot_size + assignment_size,
        'routing_start': depot_size + assignment_size,
        'routing_end': total_size,
        'total_size': total_size
    }

def decode_position_to_solution(position, encoding, instance):
    """Decode continuous position vector to LRP solution"""
    
    # Extract depot priorities
    depot_priorities = position[encoding['depot_start']:encoding['depot_end']]
    
    # Select depots to open (top priority depots)
    depot_indices = np.argsort(-depot_priorities)  # Descending order
    
    # Open at least one depot, possibly more based on priorities
    opened_depots = []
    for idx in depot_indices:
        depot = instance.depots[idx]
        opened_depots.append(depot)
        # Simple heuristic: open top 1-2 depots based on priority gap
        if len(opened_depots) >= 1 and idx > 0:
            if depot_priorities[idx] < depot_priorities[depot_indices[0]] * 0.7:
                break
        if len(opened_depots) >= 2:  # Limit to max 2 depots for this instance
            break
    
    # Extract customer assignment priorities
    assignment_priorities = position[encoding['assignment_start']:encoding['assignment_end']]
    
    # Assign customers to depots based on priorities and distances
    customer_assignments = {}
    for i, customer in enumerate(instance.customers):
        # Find best depot for this customer
        best_depot = None
        best_score = -float('inf')
        
        for depot in opened_depots:
            # Score combines assignment priority and distance
            depot_idx = instance.depots.index(depot)
            priority_score = assignment_priorities[i] if depot_idx == 0 else assignment_priorities[i] * 0.8
            distance = instance.travel_costs.get((depot, customer), float('inf'))
            distance_score = 1.0 / (1.0 + distance)  # Inverse distance
            
            total_score = priority_score + distance_score
            
            if total_score > best_score:
                best_score = total_score
                best_depot = depot
        
        if best_depot:
            customer_assignments[customer] = best_depot
    
    # Extract routing priorities
    routing_priorities = position[encoding['routing_start']:encoding['routing_end']]
    
    # Build routes using routing priorities
    routes = {}
    vehicle_id = 1
    
    for depot in opened_depots:
        # Get customers assigned to this depot
        depot_customers = [c for c, assigned_depot in customer_assignments.items() 
                          if assigned_depot == depot]
        
        # Sort customers by routing priority
        customer_priority_pairs = [(depot_customers[i], routing_priorities[i]) 
                                  for i in range(len(depot_customers))]
        customer_priority_pairs.sort(key=lambda x: -x[1])  # Descending priority
        
        # Build routes using nearest neighbor with priority guidance
        unassigned_customers = [c for c, _ in customer_priority_pairs]
        
        while unassigned_customers and vehicle_id <= len(instance.vehicles):
            route = [depot]
            current_demand = 0
            remaining_customers = unassigned_customers.copy()
            
            while remaining_customers:
                # Find next customer based on priority and distance
                best_customer = None
                best_score = -float('inf')
                
                for customer in remaining_customers:
                    if current_demand + instance.demands[customer] <= instance.vehicle_capacity:
                        # Score based on priority and distance from current position
                        customer_idx = instance.customers.index(customer)
                        priority = routing_priorities[customer_idx]
                        distance = instance.travel_costs.get((route[-1], customer), float('inf'))
                        distance_score = 1.0 / (1.0 + distance)
                        
                        total_score = priority + distance_score
                        
                        if total_score > best_score:
                            best_score = total_score
                            best_customer = customer
                
                if best_customer:
                    route.append(best_customer)
                    current_demand += instance.demands[best_customer]
                    remaining_customers.remove(best_customer)
                    unassigned_customers.remove(best_customer)
                else:
                    break
            
            # Return to depot
            route.append(depot)
            
            if len(route) > 2:  # At least depot -> customer -> depot
                routes[vehicle_id] = route
                vehicle_id += 1
            else:
                break
    
    return {
        'opened_depots': opened_depots,
        'customer_assignments': customer_assignments,
        'routes': routes
    }

print("Encoding and decoding functions implemented")

Encoding and decoding functions implemented


In [3]:
def calculate_solution_cost(solution, instance):
    """Calculate total cost of a decoded solution"""
    if not solution['opened_depots']:
        return float('inf')
    
    # Fixed depot costs
    fixed_cost = sum(instance.depot_costs[j] for j in solution['opened_depots'])
    
    # Routing costs
    routing_cost = 0
    for route in solution['routes'].values():
        for i in range(len(route) - 1):
            u, v = route[i], route[i + 1]
            routing_cost += instance.travel_costs.get((u, v), float('inf'))
    
    total_cost = fixed_cost + routing_cost
    return total_cost, fixed_cost, routing_cost

def evaluate_fitness(wolf, encoding, instance):
    """Evaluate fitness of a wolf (decode position and calculate cost)"""
    solution = decode_position_to_solution(wolf.position, encoding, instance)
    total_cost, fixed_cost, routing_cost = calculate_solution_cost(solution, instance)
    
    wolf.solution = solution
    wolf.fitness = total_cost
    
    return total_cost

def initialize_wolf_pack(population_size, encoding, instance):
    """Initialize the wolf pack with random positions"""
    wolves = []
    
    for _ in range(population_size):
        # Random position in [0, 1] range
        position = np.random.uniform(0, 1, encoding['total_size'])
        wolf = Wolf(position=position, fitness=float('inf'), solution={})
        evaluate_fitness(wolf, encoding, instance)
        wolves.append(wolf)
    
    return wolves

print("Fitness evaluation and initialization functions implemented")

Fitness evaluation and initialization functions implemented


In [4]:
def update_social_hierarchy(wolves):
    """Update wolf social hierarchy (alpha, beta, delta)"""
    # Sort wolves by fitness (lower cost is better)
    sorted_wolves = sorted(wolves, key=lambda w: w.fitness)
    
    # Reset all ranks to omega
    for wolf in wolves:
        wolf.rank = 'omega'
    
    # Assign leadership ranks
    if len(sorted_wolves) >= 1:
        sorted_wolves[0].rank = 'alpha'
    if len(sorted_wolves) >= 2:
        sorted_wolves[1].rank = 'beta'
    if len(sorted_wolves) >= 3:
        sorted_wolves[2].rank = 'delta'
    
    return sorted_wolves[0], sorted_wolves[1] if len(sorted_wolves) >= 2 else None, sorted_wolves[2] if len(sorted_wolves) >= 3 else None

def encircling_prey(wolf, alpha, beta, delta, a, encoding):
    """Update wolf position by encircling prey"""
    new_position = wolf.position.copy()
    
    for i in range(encoding['total_size']):
        # Position update with respect to alpha, beta, and delta
        r1, r2 = np.random.random(), np.random.random()
        A1, C1 = 2 * a * r1 - a, 2 * r2
        
        if alpha:
            D_alpha = abs(C1 * alpha.position[i] - wolf.position[i])
            X1 = alpha.position[i] - A1 * D_alpha
        else:
            X1 = wolf.position[i]
        
        r1, r2 = np.random.random(), np.random.random()
        A2, C2 = 2 * a * r1 - a, 2 * r2
        
        if beta:
            D_beta = abs(C2 * beta.position[i] - wolf.position[i])
            X2 = beta.position[i] - A2 * D_beta
        else:
            X2 = wolf.position[i]
        
        r1, r2 = np.random.random(), np.random.random()
        A3, C3 = 2 * a * r1 - a, 2 * r2
        
        if delta:
            D_delta = abs(C3 * delta.position[i] - wolf.position[i])
            X3 = delta.position[i] - A3 * D_delta
        else:
            X3 = wolf.position[i]
        
        # Average position of leaders
        new_position[i] = (X1 + X2 + X3) / 3.0
        
        # Ensure position stays within bounds [0, 1]
        new_position[i] = np.clip(new_position[i], 0, 1)
    
    return new_position

def grey_wolf_optimizer(instance, population_size=30, max_generations=100):
    """Complete Grey Wolf Optimizer algorithm"""
    print(f"Running Grey Wolf Optimizer with {population_size} wolves for {max_generations} generations")
    
    # Initialize encoding and wolf pack
    encoding = encode_lrp_solution(instance)
    wolves = initialize_wolf_pack(population_size, encoding, instance)
    
    # Tracking variables
    best_costs = []
    alpha_costs = []
    beta_costs = []
    delta_costs = []
    diversity_scores = []
    
    # Main optimization loop
    for generation in range(max_generations):
        # Update social hierarchy
        alpha, beta, delta = update_social_hierarchy(wolves)
        
        # Update parameter a (linearly decreases from 2 to 0)
        a = 2 - generation * (2 / max_generations)
        
        # Update positions of omega wolves
        for wolf in wolves:
            if wolf.rank == 'omega':
                new_position = encircling_prey(wolf, alpha, beta, delta, a, encoding)
                wolf.position = new_position
                evaluate_fitness(wolf, encoding, instance)
        
        # Re-evaluate all wolves and update hierarchy
        alpha, beta, delta = update_social_hierarchy(wolves)
        
        # Track statistics
        best_costs.append(alpha.fitness)
        alpha_costs.append(alpha.fitness)
        beta_costs.append(beta.fitness if beta else float('inf'))
        delta_costs.append(delta.fitness if delta else float('inf'))
        
        # Calculate population diversity
        positions = np.array([w.position for w in wolves])
        diversity = np.mean(np.std(positions, axis=0))
        diversity_scores.append(diversity)
        
        if (generation + 1) % 20 == 0:
            print(f"Generation {generation + 1}: Alpha cost = ${alpha.fitness:.2f}, Diversity = {diversity:.4f}")
    
    return alpha, wolves, best_costs, alpha_costs, beta_costs, delta_costs, diversity_scores

print("Grey Wolf Optimizer algorithm implemented")

Grey Wolf Optimizer algorithm implemented


In [ ]:
try:
    # Run Grey Wolf Optimizer
    alpha_wolf, wolf_pack, best_costs, alpha_costs, beta_costs, delta_costs, diversity_scores = grey_wolf_optimizer(
        instance, population_size=30, max_generations=100
    )
    
    print("\n" + "="*60)
    print("GREY WOLF OPTIMIZER RESULTS")
    print("="*60)
    print(f"Best Solution Cost: ${alpha_wolf.fitness:.2f}")
    print(f"Alpha Wolf Rank: {alpha_wolf.rank}")
    print(f"Opened Depots: {alpha_wolf.solution['opened_depots']}")
    print(f"Customer Assignments: {alpha_wolf.solution['customer_assignments']}")
    print(f"Number of Routes: {len(alpha_wolf.solution['routes'])}")
    
    # Calculate detailed costs
    total_cost, fixed_cost, routing_cost = calculate_solution_cost(alpha_wolf.solution, instance)
    print(f"\nCost Breakdown:")
    print(f"Fixed Depot Costs: ${fixed_cost:.2f}")
    print(f"Routing Costs: ${routing_cost:.2f}")
    print(f"Total Cost: ${total_cost:.2f}")
    
    # Show route details
    print(f"\nRoute Details:")
    for vehicle_id, route in alpha_wolf.solution['routes'].items():
        route_demand = sum(instance.demands[i] for i in route if i in instance.customers)
        utilization = (route_demand / instance.vehicle_capacity) * 100
        route_cost = sum(instance.travel_costs.get((route[i], route[i+1]), 0) for i in range(len(route)-1))
        print(f"Route {vehicle_id}: {route} (Demand: {route_demand}, Utilization: {utilization:.1f}%, Cost: ${route_cost:.2f})")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    def analyze_gwo_convergence(best_costs, alpha_costs, beta_costs, delta_costs, diversity_scores):
        """Analyze Grey Wolf Optimizer convergence behavior"""
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        generations = range(1, len(best_costs) + 1)
        
        # Plot 1: Convergence of leaders
        ax1.plot(generations, alpha_costs, 'r-', linewidth=2, label='Alpha')
        ax1.plot(generations, beta_costs, 'b-', linewidth=2, label='Beta')
        ax1.plot(generations, delta_costs, 'g-', linewidth=2, label='Delta')
        ax1.set_xlabel('Generation')
        ax1.set_ylabel('Solution Cost ($)')
        ax1.set_title('Social Hierarchy Convergence')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Population diversity over time
        ax2.plot(generations, diversity_scores, 'purple', linewidth=2)
        ax2.set_xlabel('Generation')
        ax2.set_ylabel('Population Diversity')
        ax2.set_title('Population Diversity Evolution')
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Improvement rate
        improvement_rate = []
        for i in range(1, len(best_costs)):
            if best_costs[i-1] > 0:
                improvement = (best_costs[i-1] - best_costs[i]) / best_costs[i-1] * 100
                improvement_rate.append(improvement)
            else:
                improvement_rate.append(0)
        
        ax3.plot(range(2, len(best_costs) + 1), improvement_rate, 'orange', linewidth=2)
        ax3.set_xlabel('Generation')
        ax3.set_ylabel('Improvement Rate (%)')
        ax3.set_title('Solution Improvement Rate')
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Exploration vs Exploitation
        a_values = [2 - i * (2 / len(best_costs)) for i in range(len(best_costs))]
        ax4.plot(generations, a_values, 'brown', linewidth=2)
        ax4.set_xlabel('Generation')
        ax4.set_ylabel('Control Parameter a')
        ax4.set_title('Exploration-Exploitation Balance')
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print convergence statistics
        print("\nGrey Wolf Optimizer Convergence Analysis:")
        print(f"Initial Alpha Cost: ${alpha_costs[0]:.2f}")
        print(f"Final Alpha Cost: ${alpha_costs[-1]:.2f}")
        print(f"Total Improvement: ${alpha_costs[0] - alpha_costs[-1]:.2f} ({((alpha_costs[0] - alpha_costs[-1]) / alpha_costs[0] * 100):.1f}%)")
        print(f"Average Diversity: {np.mean(diversity_scores):.4f}")
        print(f"Final Diversity: {diversity_scores[-1]:.4f}")
        
        # Find convergence point (when improvement < 1% for 10 consecutive generations)
        convergence_point = None
        for i in range(len(improvement_rate) - 10):
            if all(abs(improvement) < 1 for improvement in improvement_rate[i:i+10]):
                convergence_point = i + 2  # +2 because improvement_rate starts from generation 2
                break
        
        if convergence_point:
            print(f"Convergence Point: Generation {convergence_point}")
        else:
            print("No clear convergence point detected within maximum generations")
    
    # Analyze convergence
    analyze_gwo_convergence(best_costs, alpha_costs, beta_costs, delta_costs, diversity_scores)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'best_costs' is not defined...]

In [ ]:
try:
    def compare_population_sizes(instance, population_sizes=[10, 20, 30, 40, 50], generations=50):
        """Compare GWO performance with different population sizes"""
        
        results = {}
        
        for pop_size in population_sizes:
            print(f"Testing population size {pop_size}...")
            alpha, wolves, costs, _, _, _, _ = grey_wolf_optimizer(
                instance, population_size=pop_size, max_generations=generations
            )
            
            results[pop_size] = {
                'best_cost': alpha.fitness,
                'final_cost': costs[-1],
                'improvement': costs[0] - costs[-1],
                'improvement_pct': ((costs[0] - costs[-1]) / costs[0] * 100) if costs[0] > 0 else 0
            }
        
        # Create comparison table
        df_comparison = pd.DataFrame(results).T
        print("\nPopulation Size Comparison:")
        print(df_comparison.round(2))
        
        # Visualize comparison
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Best cost vs population size
        ax1.plot(population_sizes, [results[p]['best_cost'] for p in population_sizes], 'bo-', linewidth=2, markersize=8)
        ax1.set_xlabel('Population Size')
        ax1.set_ylabel('Best Solution Cost ($)')
        ax1.set_title('Solution Quality vs Population Size')
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Improvement vs population size
        ax2.plot(population_sizes, [results[p]['improvement'] for p in population_sizes], 'ro-', linewidth=2, markersize=8)
        ax2.set_xlabel('Population Size')
        ax2.set_ylabel('Cost Improvement ($)')
        ax2.set_title('Improvement vs Population Size')
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Improvement percentage vs population size
        ax3.plot(population_sizes, [results[p]['improvement_pct'] for p in population_sizes], 'go-', linewidth=2, markersize=8)
        ax3.set_xlabel('Population Size')
        ax3.set_ylabel('Improvement Percentage (%)')
        ax3.set_title('Relative Improvement vs Population Size')
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Efficiency (improvement per wolf)
        efficiency = [results[p]['improvement'] / p for p in population_sizes]
        ax4.plot(population_sizes, efficiency, 'mo-', linewidth=2, markersize=8)
        ax4.set_xlabel('Population Size')
        ax4.set_ylabel('Improvement per Wolf')
        ax4.set_title('Algorithm Efficiency vs Population Size')
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return results
    
    # Compare different population sizes
    population_results = compare_population_sizes(instance, population_sizes=[10, 20, 30, 40, 50], generations=50)
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    def visualize_gwo_solution(alpha_wolf, instance):
        """Visualize the best GWO solution"""
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        
        # Define positions
        positions = {
            1: (2, 8), 2: (4, 6), 3: (8, 7),  # Customers
            4: (3, 3), 5: (7, 3)              # Depots
        }
        
        # Plot nodes
        for node, pos in positions.items():
            if node in instance.customers:
                ax.scatter(pos[0], pos[1], s=300, c='lightblue', 
                          edgecolors='navy', linewidth=2, zorder=5)
                ax.annotate(f'C{node}\n(d={instance.demands[node]})', 
                           xy=pos, xytext=(pos[0]+0.3, pos[1]+0.3),
                           fontsize=10, fontweight='bold')
            elif node in instance.depots:
                color = 'lightgreen' if node in alpha_wolf.solution['opened_depots'] else 'lightgray'
                ax.scatter(pos[0], pos[1], s=400, c=color, 
                          edgecolors='darkgreen', linewidth=2, zorder=5)
                status = "OPEN" if node in alpha_wolf.solution['opened_depots'] else "CLOSED"
                cost = instance.depot_costs[node]
                ax.annotate(f'J{node-3}\n({status})\n(${cost})', 
                           xy=pos, xytext=(pos[0]-0.8, pos[1]-1.2),
                           fontsize=10, fontweight='bold')
        
        # Plot routes
        colors = ['red', 'blue', 'green', 'orange']
        for k, route in alpha_wolf.solution['routes'].items():
            if len(route) > 1:
                color = colors[k % len(colors)]
                for i in range(len(route) - 1):
                    u, v = route[i], route[i + 1]
                    u_pos, v_pos = positions[u], positions[v]
                    ax.plot([u_pos[0], v_pos[0]], [u_pos[1], v_pos[1]], 
                           color=color, linewidth=2, alpha=0.7, 
                           label=f'Vehicle {k}' if i == 0 else '')
                    
                    # Add arrow
                    mid_x = (u_pos[0] + v_pos[0]) / 2
                    mid_y = (u_pos[1] + v_pos[1]) / 2
                    dx = v_pos[0] - u_pos[0]
                    dy = v_pos[1] - u_pos[1]
                    ax.annotate('', xy=(mid_x + dx*0.1, mid_y + dy*0.1), 
                               xytext=(mid_x - dx*0.1, mid_y - dy*0.1),
                               arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
        
        # Plot assignments
        for customer, depot in alpha_wolf.solution['customer_assignments'].items():
            if depot in alpha_wolf.solution['opened_depots']:
                cust_pos = positions[customer]
                depot_pos = positions[depot]
                ax.plot([cust_pos[0], depot_pos[0]], [cust_pos[1], depot_pos[1]], 
                       'k--', alpha=0.3, linewidth=1)
        
        ax.set_xlim(-1, 10)
        ax.set_ylim(0, 10)
        ax.set_xlabel('X Coordinate', fontsize=12)
        ax.set_ylabel('Y Coordinate', fontsize=12)
        ax.set_title(f'Grey Wolf Optimizer Solution\nTotal Cost: ${alpha_wolf.fitness:.2f}', 
                     fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()
    
    # Visualize best solution
    visualize_gwo_solution(alpha_wolf, instance)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'alpha_wolf' is not defined...]

### Why this Tier exists vs earlier Tiers
The Grey Wolf Optimizer addresses limitations of both exact and heuristic approaches:

**Tier 1 Limitations:**
- ❌ Exponential complexity for large instances
- ❌ Impractical for real-world problem sizes

**Tier 2 Limitations:**
- ❌ Limited exploration - can get stuck in local optima
- ❌ Single-solution focus - no population-based search
- ❌ Greedy construction may miss better regions

**Tier 3 Advantages:**
- ✅ **Population-based search** - explores multiple solution regions simultaneously
- ✅ **Social hierarchy** - alpha, beta, delta wolves guide search effectively
- ✅ **Balance exploration/exploitation** - adaptive parameter control
- ✅ **Nature-inspired intelligence** - emergent collective behavior
- ✅ **Better global optimization** - avoids local optima traps

### Pros / Cons vs earlier Tiers

**Pros:**
- ✅ **Superior exploration** - population search vs single-solution methods
- ✅ **Adaptive behavior** - dynamic balance between exploration and exploitation
- ✅ **Leadership guidance** - alpha, beta, delta wolves provide intelligent search direction
- ✅ **Emergent intelligence** - collective behavior finds better solutions
- ✅ **Scalable to complex problems** - handles high-dimensional search spaces
- ✅ **Less parameter sensitivity** - fewer parameters than GRASP

**Cons:**
- ❌ **Higher computational cost** - population evaluation vs single solution
- ❌ **Complex encoding** - continuous-to-discrete mapping challenges
- ❌ **Convergence uncertainty** - may require many generations
- ❌ **Memory intensive** - stores entire population

### When to use this Tier
- **Complex optimization landscapes** with many local optima
- **Large-scale instances** where heuristics get stuck
- **Multi-modal problems** requiring global search capability
- **Quality-critical applications** where solution optimality matters
- **Research and development** testing new optimization approaches
- **Benchmark studies** comparing metaheuristic performance

### Key Grey Wolf Optimizer Innovations

1. **Social Hierarchy**: Alpha, beta, delta wolves provide intelligent search guidance

2. **Encircling Prey**: Mathematical modeling of wolf hunting behavior

3. **Adaptive Parameters**: Control parameter 'a' balances exploration vs exploitation

4. **Population Diversity**: Maintains search space exploration throughout optimization

5. **Continuous Encoding**: Maps discrete LRP to continuous optimization space

6. **Collective Intelligence**: Emergent behavior finds solutions beyond individual capability

The Grey Wolf Optimizer represents a sophisticated approach to Location-Routing Problems, leveraging nature-inspired collective intelligence to find high-quality solutions in complex search spaces where traditional methods may fail.